# ARM-Gym GRPO Training (Apple Silicon M3 Pro)

Train Qwen2.5-Coder-7B to generate optimized AArch64 assembly that beats gcc -O3.

**Hardware**: MacBook M3 Pro 18GB unified memory, MPS backend
**Requirements**: Homebrew, Python 3.10+, ~15GB free RAM
**Time**: ~60-120 min for 50 steps on M3 Pro

## 1. Install ARM cross-compilation toolchain (Homebrew LLVM)

In [1]:
import subprocess, os, shutil

llvm_bin = "/opt/homebrew/opt/llvm/bin"

if not shutil.which("llvm-mca") and llvm_bin not in os.environ.get("PATH", ""):
    print("Installing LLVM via Homebrew (may take a few minutes)...")
    r = subprocess.run(["brew", "install", "llvm"], capture_output=True, text=True)
    if r.returncode != 0:
        print("brew install llvm failed:")
        print(r.stderr[-1000:])
    else:
        print(r.stdout[-500:])

if llvm_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = llvm_bin + ":" + os.environ["PATH"]

for tool in ["clang", "llvm-mca"]:
    path = shutil.which(tool)
    print(f"{tool}: {path or 'NOT FOUND'}")

print("toolchain done")

Installing LLVM via Homebrew (may take a few minutes)...
cOS unless you know what you're doing.

llvm is keg-only, which means it was not symlinked into /opt/homebrew,
because macOS already provides this software and installing another version in
parallel can cause all kinds of trouble.

If you need to have llvm first in your PATH, run:
  echo 'export PATH="/opt/homebrew/opt/llvm/bin:$PATH"' >> ~/.zshrc

For compilers to find llvm you may need to set:
  export LDFLAGS="-L/opt/homebrew/opt/llvm/lib"
  export CPPFLAGS="-I/opt/homebrew/opt/llvm/include"

clang: /opt/homebrew/opt/llvm/bin/clang
llvm-mca: /opt/homebrew/opt/llvm/bin/llvm-mca
toolchain done


## 2. Install training stack

In [2]:
import subprocess, sys
cmds = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
    [sys.executable, "-m", "pip", "install", "-q",
     "torch", "transformers", "trl>=0.16", "peft", "accelerate",
     "datasets", "pydantic", "numpy", "matplotlib", "httpx"],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"WARN: {' '.join(cmd[:5])}... failed")
        print(r.stderr[-500:])
    else:
        print(f"OK: {' '.join(cmd[:5])}...")
print("deps done")

OK: /Users/vetri/.pyenv/versions/3.10.11/bin/python -m pip install -q...
OK: /Users/vetri/.pyenv/versions/3.10.11/bin/python -m pip install -q...
deps done


## 3. Set up arm_gym source path

In [3]:
import sys, os
from pathlib import Path

here = Path(os.getcwd())
for root in [here.parent, here, here.parent.parent]:
    if (root / "arm_gym" / "__init__.py").exists():
        sys.path.insert(0, str(root))
        print(f"arm_gym source at: {root}")
        break
else:
    raise RuntimeError("arm_gym not found - run notebook from arm-gym/ or arm-gym/colab/ directory")

arm_gym source at: /Users/vetri/Downloads/Personal_Project/RL/Finale/arm-gym


## 4. Smoke test - verify toolchain + reward loop

In [4]:
import os, sys

llvm_bin = "/opt/homebrew/opt/llvm/bin"
if llvm_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = llvm_bin + ":" + os.environ["PATH"]

from arm_gym.compile_baseline import detect_toolchain
tc = detect_toolchain()
print(f"clang={tc.clang} gcc={tc.gcc_aarch64} mca={tc.mca} mcpu={tc.mcpu}")
assert tc.ready(), "toolchain not ready - rerun cell 1"
assert tc.mca, "llvm-mca not found - rerun cell 1"

from arm_gym.kernels import summary, generate_variants
s = summary()
print(f"templates={s['templates']} variants={s['variants']}")

from arm_gym.compile_baseline import compile_to_asm
v = next(generate_variants("vec_add"))
asm = compile_to_asm(v.c_source, tc)
print(f"compiled vec_add variant, asm length={len(asm)}")

from arm_gym.mca import run_mca
rep = run_mca(asm, tc.mca, tc.mcpu)
print(f"MCA: cycles={rep.total_cycles} ipc={rep.ipc:.2f}")
print("smoke OK")

clang=clang gcc=None mca=llvm-mca mcpu=neoverse-v3
templates=15 variants=523
compiled vec_add variant, asm length=644
MCA: cycles=291 ipc=3.78
smoke OK


## 5. Device detection (MPS)

In [5]:
import torch

if torch.backends.mps.is_available():
    device = "mps"
    print(f"MPS available - Apple Silicon GPU acceleration enabled")
else:
    device = "cpu"
    print("WARNING: MPS not available, falling back to CPU (training will be very slow)")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"float16 support: {torch.tensor([1.0], dtype=torch.float16).device}")

MPS available - Apple Silicon GPU acceleration enabled
PyTorch version: 2.10.0
Device: mps
float16 support: cpu


## 6. Build training dataset

In [6]:
import os

llvm_bin = "/opt/homebrew/opt/llvm/bin"
if llvm_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = llvm_bin + ":" + os.environ["PATH"]

from arm_gym.compile_baseline import detect_toolchain
from kaggle.dataset import DatasetConfig, build as build_dataset

tc = detect_toolchain()

DIFFICULTY = 1
MAX_TRAIN = 128
MAX_EVAL = 16

cfg = DatasetConfig(max_train=MAX_TRAIN, max_eval=MAX_EVAL, difficulty_max=DIFFICULTY)
train_ds, eval_ds, lookup = build_dataset(tc, cfg, tokenizer=None)
print(f"train={len(train_ds)} eval={len(eval_ds)} lookup={len(lookup)}")
print(f"sample prompt length: {len(train_ds[0]['prompt'])} chars")
print(f"sample variant_id: {train_ds[0]['variant_id']}")

/Users/vetri/.pyenv/versions/3.10.11/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train=128 eval=16 lookup=144
sample prompt length: 1207 chars
sample variant_id: vec_add_39d0a6df285a


## 7. GRPO Training

Plain PEFT LoRA on MPS (Apple Silicon M3 Pro).
- lora_rank=8, lora_alpha=16
- num_generations=2, temperature=0.5
- 50 steps (~60-120 min on M3 Pro)

> **Memory note**: 7B in float16 uses ~14GB. Ensure no other large apps are open.
> For a faster smoke test, switch MODEL_ID to `Qwen/Qwen2.5-Coder-1.5B-Instruct`.

In [7]:
import os, sys, time, csv, re
import torch
from pathlib import Path

llvm_bin = "/opt/homebrew/opt/llvm/bin"
if llvm_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = llvm_bin + ":" + os.environ["PATH"]

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_RANK = 8
LORA_ALPHA = 16
STEPS = 50
NUM_GENERATIONS = 2
MAX_PROMPT_LEN = 512
MAX_COMPLETION_LEN = 256
LR = 1e-6
BATCH_SIZE = 1
GRAD_ACCUM = 4
OUT_DIR = "runs/m3-grpo"

os.makedirs(OUT_DIR, exist_ok=True)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model (float16, ~14GB)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": device},
    low_cpu_mem_usage=True,
)
lora = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0,
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("model ready")

from arm_gym.compile_baseline import detect_toolchain
from kaggle.dataset import DatasetConfig, build as build_dataset

tc = detect_toolchain()
ds_cfg = DatasetConfig(max_train=128, max_eval=16, difficulty_max=1)
train_ds, eval_ds, _ = build_dataset(tc, ds_cfg, tokenizer=tokenizer)
print(f"dataset: train={len(train_ds)} eval={len(eval_ds)}")

from kaggle.reward_fn import syntax_reward, correctness_reward, speedup_reward
from trl import GRPOConfig, GRPOTrainer

grpo_params = dict(
    output_dir=OUT_DIR,
    max_steps=STEPS,
    learning_rate=LR,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPLETION_LEN,
    gradient_checkpointing=False,
    bf16=False,
    fp16=False,
    max_grad_norm=0.1,
    temperature=0.5,
    beta=0.0,
    epsilon=0.2,
    remove_unused_columns=False,
    logging_steps=1,
    save_steps=25,
    save_total_limit=2,
    report_to="none",
    no_cuda=True,
)

while True:
    try:
        gcfg = GRPOConfig(**grpo_params)
        break
    except TypeError as exc:
        m = re.search(r"unexpected keyword argument '(\w+)'", str(exc))
        if not m:
            raise
        dropped = m.group(1)
        print(f"TRL compat: dropping {dropped!r}")
        grpo_params.pop(dropped, None)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[syntax_reward, correctness_reward, speedup_reward],
    args=gcfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print(f"GRPO training: {STEPS} steps, lr={LR}, gen={NUM_GENERATIONS}")
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"training done in {elapsed/60:.1f} min")

log_path = Path(OUT_DIR) / "log.csv"
history = getattr(trainer.state, "log_history", [])
if history:
    with open(log_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        w.writeheader()
        w.writerows(history)
    print(f"saved log to {log_path} ({len(history)} rows)")

model.save_pretrained(f"{OUT_DIR}/lora-adapter")
tokenizer.save_pretrained(f"{OUT_DIR}/lora-adapter")
print(f"saved LoRA adapter to {OUT_DIR}/lora-adapter")

device: mps
Loading tokenizer...


Loading model (float16, ~14GB)...


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files: 100%|██████████| 4/4 [08:58<00:00, 134.51s/it]


RuntimeError: Invalid buffer size: 14.19 GiB

## 8. Generate training evidence plots

In [ ]:
import os, sys, glob
import matplotlib
matplotlib.use("Agg")
from pathlib import Path

from kaggle.plot_curves import (
    load_rows, plot_training_loss, plot_reward_curve,
    plot_correctness_rate, plot_before_after,
)

log_path = Path("runs/m3-grpo/log.csv")
out_path = Path("artifacts/plots")
out_path.mkdir(parents=True, exist_ok=True)

if log_path.exists():
    rows = load_rows(log_path)
    print(f"loaded {len(rows)} log rows")
    plot_training_loss(rows, out_path / "training_loss.png")
    plot_reward_curve(rows, out_path / "reward_curve.png")
    plot_correctness_rate(rows, out_path / "correctness_rate.png")
    print("generated 3 training curve plots")
else:
    print(f"WARNING: {log_path} not found - run training cell first")

plot_before_after(out_path / "before_after_kernel.png")

for png in sorted(glob.glob(str(out_path / "*.png"))):
    print(f"saved: {png}")

## 9. View results

In [ ]:
import os, glob
from pathlib import Path

out_dir = Path("runs/m3-grpo")
print(f"Run directory: {out_dir.resolve()}")
for f in sorted(out_dir.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.relative_to(out_dir)}  ({size/1024:.1f} KB)")

plots = sorted(glob.glob("artifacts/plots/*.png"))
print(f"\nPlots ({len(plots)}):")
for p in plots:
    print(f"  {os.path.abspath(p)}")

## 10. (Optional) Push LoRA adapter to HuggingFace Hub

Uncomment and set your HF token to push the trained adapter.

In [ ]:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# model.push_to_hub("your-username/arm-gym-grpo-lora")
# tokenizer.push_to_hub("your-username/arm-gym-grpo-lora")
# print("pushed to HF Hub")
print("uncomment above to push adapter to HF Hub")